In [25]:
import requests
import os
import pandas as pd
import numpy as np
import holidays
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Chargement des données et pré-traitement

In [26]:
def load_data(url):
    response = requests.get(url)

    #On recup les données et on les met sous forme csv
    if response.status_code // 100 == 2:
        data = response.json()
        availableBikeNumber= data.get("availableBikeNumber")
        values = availableBikeNumber.get("values")
        data_clean = []
        for value in values:
            bikeNumber = value[0]
            date = value[1]
            line = {"availableBikeNumber": bikeNumber, "date": date}
            data_clean.append(line)
        df = pd.DataFrame(data_clean)
        os.makedirs("data", exist_ok=True)
        df.to_csv("data/sample.csv", index=False)
        print("Données récupérées")

        return df

    else :
        print(f"Erreur : {response.status_code}")
        return None



In [27]:
#df = pd.read_csv("data/sample.csv")

def clean_data(df):
    df["date"] = pd.to_datetime(df["date"]).dt.tz_convert('Europe/Paris').dt.tz_localize(None)
    df["availableBikeNumber"] = df["availableBikeNumber"].astype(int)

    #Nettoyage et rajout de colonnes pour avoir + d'informations
    df["time"] = df["date"].dt.time
    df["hour"] = df["date"].dt.hour
    df["dayOfWeek"] = df["date"].dt.dayofweek
    df["isWeekend"] = df["date"].dt.dayofweek.isin([5,6]).astype(int)
    annees = df["date"].dt.year.unique()
    holidays_france = holidays.France(years=annees)
    df["isHolidays"] = df["date"].dt.date.isin(holidays_france).astype(int)

    return df

#pd.set_option('display.width', 1000)
#print(df.head())

# Entraînement du modèle

In [28]:
#Crée le dataframe où sont placés nos résultats
def create_results(X_test, y_test, predictions):
    df_resultats = X_test.copy()
    df_resultats["velosReels"] = y_test
    df_resultats["Erreur absolue"] = np.round(abs(y_test - predictions), 2)
    df_resultats["velosPredits"] = np.round(predictions, 0).astype(int)
    return df_resultats

#Entraîne le modèle
def evaluate_model(pipeline, X, y, n_splits=5):
    #kfold temporel
    tscv = TimeSeriesSplit(n_splits=n_splits)

    mae_scores = []
    rmse_scores = []
    liste_df_resultats = []

    #Entraînement des données et prédictions
    for fold, (train_index, test_index) in enumerate(tscv.split(X), 1):
        #Séparation des données
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]

        #Entraînement du modèle sur données train
        pipeline.fit(X_train, y_train)

        #Prédiction des données test
        predictions = pipeline.predict(X_test)

        #Calcul des indicateurs de performance
        mae = mean_absolute_error(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))

        mae_scores.append(mae)
        rmse_scores.append(rmse)

        #On place les résultats dans une liste
        df_resultats = create_results(X_test, y_test, predictions)
        liste_df_resultats.append(df_resultats)

        print(f"Tour {fold} — MAE : {mae:.2f} | RMSE : {rmse:.2f}")

    #On regroupe tous les résultats
    df_resultats_finaux = pd.concat(liste_df_resultats, ignore_index=True)

    print("\n--- Résultats ---")
    print(f"MAE Moyenne : {np.mean(mae_scores):.2f}")
    print(f"RMSE Moyenne : {np.mean(rmse_scores):.2f}")

    return df_resultats_finaux

#Création du pipeline
pipeline = Pipeline([("scaler", StandardScaler()), ("model", RandomForestRegressor(random_state=42))])



# Main

In [30]:
#URL de l'historique des places Velomagg de la station 001
url = "https://portail-api-data.montpellier.fr/ngsi-ld/v1/temporal/entities/urn%3Angsi-ld%3Astation%3A001?format=temporalValues&timerel=after&timeAt=2025-12-31T23%3A59%3A59Z"


print("Chargement des données...")
df = load_data(url)

print("Nettoyage des données...")
df_clean = clean_data(df)

features = ["hour", "dayOfWeek", "isHolidays", "isWeekend"]
X = df[features]
y = df["availableBikeNumber"]

print("Entraînement du modèle...")
df_resultats_finaux = evaluate_model(pipeline, X, y, n_splits=5)


#Importation csv pour PowerBI de nos résultats
os.makedirs("results", exist_ok=True)
df_resultats_finaux.to_csv("results/resultats.csv", index=False, encoding="utf-8-sig")
print("Résultats enregistrés\n")

#Affichage de nos résultats
pd.set_option('display.width', 1000)
print(df_resultats_finaux.head())


Chargement des données...
Données récupérées
Nettoyage des données...
Entraînement du modèle...
Tour 1 — MAE : 3.39 | RMSE : 3.65
Tour 2 — MAE : 2.71 | RMSE : 3.01
Tour 3 — MAE : 1.27 | RMSE : 1.62
Tour 4 — MAE : 2.66 | RMSE : 2.96
Tour 5 — MAE : 2.85 | RMSE : 3.31

--- Résultats ---
MAE Moyenne : 2.58
RMSE Moyenne : 2.91
Résultats enregistrés

   hour  dayOfWeek  isHolidays  isWeekend  velosReels  Erreur absolue  velosPredits
0     4          4           0          0           2             0.0             2
1     4          4           0          0           2             0.0             2
2     4          4           0          0           2             0.0             2
3     4          4           0          0           2             0.0             2
4     4          4           0          0           2             0.0             2
